# 00 - Design do dataset

Este notebook documenta o **design** da base de dados que vamos construir.

O objetivo aqui é construir um dataset observacional em que cada linha corresponde a uma *configuração experimental* extraída de um artigo científico que aplica (ou compara) estratégias de balanceamento de classes em problemas de classificação supervisionada. A partir dessa base, queremos estimar o **efeito causal médio** de adotar uma estratégia de balanceamento sobre o desempenho do classificador (macro-F1 como métrica primária, dado ser mais comum), além de explorar heterogeneidade do efeito por características do dataset e do modelo.

A pergunta causal é intervencional: $P(Y \mid \mathrm{do}(X))$, onde $X$ é a estratégia de balanceamento e $Y$ é o desempenho. 

## Arquitetura do pipeline

A construção da base segue uma **cascata de filtros**, do mais barato e permissivo ao mais caro e seletivo:

```
[~250M+ papers]   Universo (OpenAlex/S2/arXiv combinados)
      │
      │ Filtro 1: keywords + categorias (grátis)
      ▼
[~5k–20k papers]  Candidatos por triagem textual
      │
      │ Filtro 2: ranking por embedding semântico (barato, local)
      ▼
[~2k–5k papers]   Top-K mais relevantes
      │
      │ Filtro 3: classificador LLM sobre o abstract (médio)
      ▼
[~500–1500 papers] Candidatos com PDF acessível
      │
      │ Download + parse de PDFs
      ▼
[~300–1000 papers] Full text disponível
      │
      │ Extração estruturada via LLM (mais caro)
      ▼
[base bruta]      Configurações experimentais extraídas (1 paper → N linhas)
      │
      │ Normalização (regras + LLM auxiliar)
      ▼
[base normalizada] Pronta para análise causal
```

### Mapeamento notebook → estágio

| Notebook | Estágio |
|----------|---------|
| `01_extracao_papers.ipynb` | Filtros 1 e 2 (discovery + ranking por embedding) |
| `02_classificacao_llm.ipynb` | Filtro 3 (classificador LLM sobre abstract) |
| `03_download_pdfs.ipynb` | Download + parsing de PDFs open access |
| `04_extracao_estruturada.ipynb` | Extração via LLM (campos brutos) |
| `05_normalizacao.ipynb` | Normalização para schema canônico |

## Decisões metodológicas

Decisões fechadas até aqui:

1. **Unidade de análise:** configuração experimental (`paper × dataset × modelo × estratégia`). Um paper pode contribuir com várias linhas.
2. **Estratégia de coleta:** *extrai tudo, decide depois*. Todo experimento relatado entra na base bruta, mesmo papers sem comparação interna. A partição **within-paper** (papers com ≥2 estratégias no mesmo dataset+modelo) e **cross-paper** emerge naturalmente na análise. A análise principal será within-paper (pareada, menos confundimento); cross-paper entra como secundária/exploratória.
3. **Tratamento $X$:** categórico amplo (`none / oversampling / undersampling / hybrid / cost_sensitive / data_augmentation / ensemble_based / threshold_moving / generative / other`). Ver tipologia abaixo.
4. **Desfecho $Y$:** macro-F1 prioritário, mas guardamos **todas** as métricas que o paper reportar (F1 binário/weighted/micro, balanced accuracy, AUROC, AUPRC, MCC, G-mean, TPR gap, accuracy). A escolha da métrica de análise é deslocada para a fase de modelagem.
5. **Definição de baseline:** "sem intervenção" só conta se o paper menciona explicitamente (`baseline / no balancing / vanilla / unmodified`).
6. **Fontes de descoberta:** união de **arXiv (API) + OpenAlex + Semantic Scholar**. Restrição ao subconjunto com PDF open access para garantir reprodutibilidade.
7. **Modelos LLM:** classificador de abstract com modelo barato; extração estruturada com modelo intermediário.

## Schema da base bruta - tabela `papers`

Uma linha por artigo, indexada por um identificador interno. Mantém o estado de progresso de cada paper no pipeline.

| Campo | Tipo | Descrição |
|-------|------|-----------|
| `paper_id` | str | ID interno (UUID ou hash do título+ano normalizados) |
| `arxiv_id` | str? | ID arXiv se disponível |
| `doi` | str? | DOI normalizado (lowercase, sem prefixo URL) |
| `s2_paper_id` | str? | ID do Semantic Scholar |
| `openalex_id` | str? | ID do OpenAlex |
| `title` | str | Título |
| `abstract` | str | Resumo |
| `authors` | list[str] | Autores |
| `year` | int | Ano de publicação |
| `venue` | str? | Venue (revista/conferência), se conhecido |
| `categories` | list[str]? | Categorias arXiv (e.g., `cs.LG`) |
| `discovered_via` | list[str] | Fontes que retornaram o paper (`arxiv`, `openalex`, `s2`) |
| `discovered_queries` | list[str] | Queries que retornaram o paper |
| `oa_pdf_url` | str? | URL do PDF open access |
| `is_oa` | bool | Tem PDF acessível? |
| `pdf_local_path` | str? | Caminho do PDF baixado |
| `emb_score` | float? | Similaridade máxima a queries-âncora (filtro 2) |
| `classifier_label` | str? | Saída do filtro 3: `relevant` / `not_relevant` / `uncertain` |
| `classifier_reasoning` | str? | Justificativa curta do classificador |
| `extraction_status` | str | `pending` / `extracted` / `failed` / `excluded` |
| `extraction_error` | str? | Mensagem de erro se falhou |

## Schema da base bruta - tabela `configurations`

Uma linha por configuração experimental extraída. Foreign key para `papers.paper_id`. Mantém os valores **literais** extraídos do paper (com sufixo `_raw`) ao lado de campos auxiliares de normalização. A regra é: **preserve a string original**, normalize depois.

| Campo | Tipo | Descrição |
|-------|------|-----------|
| `config_id` | str | ID interno da configuração |
| `paper_id` | str | FK para `papers` |
| `dataset_name_raw` | str | Nome do dataset como aparece no paper |
| `dataset_size_raw` | str? | Tamanho relatado (texto literal, e.g. "1,234 instances") |
| `dataset_num_classes_raw` | str? | Nº de classes (texto literal) |
| `dataset_imbalance_ratio_raw` | str? | IR ou descrição do desbalanceamento |
| `dataset_domain_raw` | str? | Domínio descrito (ex: "medical imaging", "credit fraud") |
| `task_type_raw` | str? | Tipo de tarefa (ex: "binary text classification") |
| `model_name_raw` | str | Modelo como descrito no paper |
| `model_hparams_raw` | str? | Hiperparâmetros relevantes, se relatados |
| `balancing_strategy_raw` | str | Estratégia textual exata (ex: "SMOTE with k=5") |
| `is_baseline_raw` | bool | O paper marca isto como baseline sem intervenção? |
| `metric_name_raw` | str | Métrica como aparece (ex: "F1-score", "macro F1") |
| `metric_value` | float | Valor numérico |
| `metric_split_raw` | str? | `test` / `val` / `cv` (texto literal) |
| `metric_aggregation_raw` | str? | `mean`, `mean±std`, `best`, etc. |
| `extracted_evidence` | str | Trecho do paper que justifica a extração (citação curta) |
| `extraction_confidence` | float | Auto-avaliação do LLM (0-1) |
| `notes` | str? | Observações livres do extrator |

## Schema da base normalizada — tabela `experiments`

Versão limpa, com vocabulário controlado, pronta para análise causal. Produzida a partir de `configurations` no notebook `05_normalizacao.ipynb`.

| Campo | Tipo | Descrição |
|-------|------|-----------|
| `experiment_id` | str | = `config_id` |
| `paper_id` | str | FK para `papers` |
| `year` | int | Ano do paper |
| `task_type` | enum | `vision` / `text` / `tabular` / `audio` / `time_series` / `medical_imaging` / `other` |
| `dataset_canonical` | str | Nome canônico (vocabulário controlado, ver abaixo) |
| `dataset_size` | int? | Tamanho numérico |
| `dataset_num_classes` | int? | Nº de classes |
| `dataset_imbalance_ratio` | float? | IR = max class size / min class size |
| `dataset_is_multilabel` | bool | Multilabel? |
| `model_family` | enum | `linear` / `tree` / `kernel` / `mlp` / `cnn` / `rnn` / `transformer` / `gbm` / `ensemble` / `gnn` / `other` |
| `model_specific` | str? | Nome específico (ex: `resnet50`, `bert-base`, `xgboost`) |
| `balancing_strategy` | enum | Categoria (ver tipologia abaixo) |
| `balancing_strategy_specific` | str? | Variante específica (ex: `smote`, `adasyn`, `focal_loss`) |
| `is_baseline` | bool | Configuração de baseline sem balanceamento |
| `metric_f1_macro` | float? | Macro-F1 (se reportado) |
| `metric_f1_weighted` | float? | Weighted-F1 |
| `metric_f1_binary` | float? | F1 binário |
| `metric_f1_micro` | float? | Micro-F1 |
| `metric_balanced_acc` | float? | Balanced accuracy |
| `metric_accuracy` | float? | Accuracy bruta |
| `metric_aucroc` | float? | AUROC |
| `metric_auprc` | float? | AUPRC |
| `metric_mcc` | float? | Matthews correlation |
| `metric_gmean` | float? | G-mean |
| `metric_tpr_gap` | float? | TPR gap |
| `metric_split` | enum | `test` / `val` / `cv` |
| `within_paper_group_id` | str? | ID do grupo `(paper, dataset, modelo)`. Linhas com mesmo group_id são comparações pareadas |
| `has_baseline_in_group` | bool | O grupo contém um baseline? |

## Tipologia de `balancing_strategy`

Usada na coluna `balancing_strategy` da base normalizada. A coluna `balancing_strategy_specific` preserva a variante exata.

| Categoria | Definição | Exemplos |
|-----------|-----------|----------|
| `none` | Nenhuma intervenção sobre a distribuição de classes ou função de perda (baseline explícito) | "no resampling", "vanilla", "unmodified" |
| `oversampling` | Aumenta a classe minoritária replicando ou sintetizando amostras | Random oversampling, SMOTE, ADASYN, Borderline-SMOTE, SVMSMOTE, K-Means SMOTE |
| `undersampling` | Reduz a classe majoritária | Random undersampling, Tomek Links, ENN, NearMiss, CNN, OSS |
| `hybrid` | Combina over + undersampling | SMOTE+Tomek, SMOTE+ENN, SMOTEENN, SMOTETomek |
| `cost_sensitive` | Modifica a função de perda ou pesos de classe sem alterar os dados | Class weights, weighted cross-entropy, focal loss, LDAM, asymmetric loss |
| `data_augmentation` | Aumento de dados class-aware (especialmente em visão/NLP) | Class-balanced mixup, CutMix balanceado, back-translation orientada |
| `ensemble_based` | Métodos de ensemble especializados em desbalanceamento | BalancedBagging, RUSBoost, EasyEnsemble, BalancedRF |
| `threshold_moving` | Ajusta o threshold de decisão pós-treino | Optimal threshold via validation, Platt scaling com calibração balanceada |
| `generative` | Geração de amostras sintéticas com modelos generativos | GAN-based oversampling (GAMO, BAGAN), VAE-based |
| `two_stage` | Treina em duas fases (representação balanceada vs. classifier) | Decoupled training, classifier retraining, cRT |
| `other` | Não se encaixa nas categorias acima | Métodos novos ou específicos de domínio |

**Regra para múltiplas estratégias simultâneas:** se o paper combina (ex: SMOTE + focal loss), criamos uma linha por estratégia primária ou marcamos como `hybrid` com `balancing_strategy_specific` descrevendo a combinação. A decisão exata fica para o notebook de normalização — por ora, registramos a string literal no `_raw`.

## Critérios de aceitação

Uma `configuration` extraída só entra na base normalizada se atender a TODOS os critérios:

1. **Identificação do dataset:** `dataset_name_raw` não-nulo, E pelo menos 2 de: `dataset_size`, `dataset_num_classes`, `dataset_imbalance_ratio`.
2. **Identificação do modelo:** classificável em uma família de `model_family` (i.e., `other` é permitido, mas modelo precisa ser nominado).
3. **Identificação da estratégia:** classificável em uma categoria de `balancing_strategy` (incluindo `none` quando o baseline é explícito).
4. **Métrica utilizável:** pelo menos um campo de métrica numérica preenchido. Para análise principal, restringimos a configurações com `metric_f1_macro`, `metric_f1_weighted`, `metric_f1_binary`, `metric_balanced_acc`, ou `metric_tpr_gap`.
5. **Split identificado:** sabemos se o valor reportado é de teste, validação ou validação cruzada.
6. **Consistência interna:** se `is_baseline=True`, então `balancing_strategy=none`.

A tabela `papers` mantém todos os papers processados, mesmo os excluídos, com `extraction_status` indicando o motivo. Isso é importante para reportar viés de seleção e auditar o pipeline.